In [1]:
import pandas as pd

In [2]:
df = pd.read_csv( "../data/processed/processed_resume.csv"
)

In [3]:
print(df.shape)

(2481, 6)


In [4]:
print(df["clean_resume"].iloc[0])

hr administrator marketing associate hr administrator summary dedicated customer service manager with 15  years of experience in hospitality and customer service management  respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service  highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq   micros opera pms   fidelio opera reservation system  ors    holidex completed courses and seminars in customer service  sales strategies  inventory control  loss prevention  safety  time management  leadership and performance assessment  experience hr adm

In [5]:
import re

In [6]:
def extract_experience(text):
    patterns = [
        r'(\d+)\+?\s+years?\s+of\s+experience',
        r'(\d+)\+?\s+years?\s+experience',
        r'(\d+)\+?\s+years?\s+in'
    ]

    for pattern in patterns:
        match = re.search(pattern, text.lower())

        if match:
            return int(match.group(1))

    return 0

In [7]:
test_text = "I have 3 years of experience in Python and SQL."

print(extract_experience(test_text))

3


In [8]:
df["years_experience"] = df["clean_resume"].apply(
    extract_experience
)

In [9]:
df[
    ["ID", "Category", "years_experience"]
].head(20)

,ID,Category,years_experience
0,16852973,HR,15
1,22323967,HR,0
2,33176873,HR,20
3,27018550,HR,0
4,17812897,HR,0
5,11592605,HR,0
6,25824789,HR,0
7,15375009,HR,0
8,11847784,HR,15
9,32896934,HR,0


In [10]:
df["years_experience"].describe()

count    2481.000000
mean        2.262797
std         5.836228
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        44.000000
Name: years_experience, dtype: float64

In [11]:
experience_check = df[
    df["years_experience"] > 0
][
    ["ID", "Category", "years_experience", "clean_resume"]
]

experience_check[
    ["ID", "Category", "years_experience"]
].head(20)

,ID,Category,years_experience
0,16852973,HR,15
2,33176873,HR,20
8,11847784,HR,15
10,29149998,HR,3
19,13879043,HR,7
21,18827609,HR,3
22,25676643,HR,9
28,18334783,HR,20
38,51769822,HR,30
46,34554248,HR,3


In [12]:
print(
    experience_check["clean_resume"].iloc[0]
)

hr administrator marketing associate hr administrator summary dedicated customer service manager with 15  years of experience in hospitality and customer service management  respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service  highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq   micros opera pms   fidelio opera reservation system  ors    holidex completed courses and seminars in customer service  sales strategies  inventory control  loss prevention  safety  time management  leadership and performance assessment  experience hr adm

In [13]:
high_experience = df[
    df["years_experience"] > 15
][
    ["ID", "Category", "years_experience", "clean_resume"]
]

high_experience[
    ["ID", "Category", "years_experience"]
]

,ID,Category,years_experience
2,33176873,HR,20
28,18334783,HR,20
38,51769822,HR,30
121,36269672,DESIGNER,44
122,90066849,DESIGNER,28
...,...,...,...
2418,10945968,AVIATION,20
2449,19284293,AVIATION,20
2450,11752500,AVIATION,20
2466,49646155,AVIATION,17


In [14]:
print(high_experience["clean_resume"].iloc[0])

hr director summary over 20 years experience in recruiting  15 plus years in human resources executive management  5 years of hris development and maintenance 4 years working in a healthcare enviroment skills recruiting fmla eeo flsa hris development benefit administration policy development web page development accomplishments kansas health institute  health outcomes for the state of kansas  1999 memberships and accolades  project management institute member  shrm  chamber of commerce  1999 friends university president s honor roll  1997 friends university dean s honor roll  student liaison for friends university topeka  member of mother to mother  member of the topeka advertising federation  several production pieces created nominated for addy awards  received recognition for outstanding customer service assistance by the state of kansas travel and tourism department   ashhra  kahhr  acm  additional information  leading change  i have been instrumental in development and implementati

In [15]:
zero_experience = df[
    df["years_experience"] == 0
]

print(zero_experience[
    ["ID", "Category", "clean_resume"]
].head(5).to_string())

         ID Category                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [16]:
import re

In [17]:
def extract_date_ranges(text):
    pattern = r'(\d{1,2})\s+(\d{4})\s+to\s+(current|\d{1,2}\s+\d{4})'

    matches = re.findall(pattern, text.lower())

    return matches

In [18]:
text = df.loc[
    df["ID"] == 22323967,
    "clean_resume"
].iloc[0]

print(extract_date_ranges(text))

[('09', '2015', 'current'), ('01', '2014', '05 2015'), ('10', '2012', '01 2014'), ('09', '2009', '10 2012')]


In [19]:
from datetime import datetime

In [20]:
def calculate_duration(start_month, start_year, end_value):
    
    start_date = datetime(
        int(start_year),
        int(start_month),
        1
    )

    if end_value == "current":
        end_date = datetime.now()
    else:
        end_month, end_year = end_value.split()
        
        end_date = datetime(
            int(end_year),
            int(end_month),
            1
        )

    months = (
        (end_date.year - start_date.year) * 12
        + (end_date.month - start_date.month)
    )

    return months

In [21]:
print(
    calculate_duration(
        "09",
        "2015",
        "current"
    )
)

132


In [29]:
def calculate_total_experience(text):
    
    date_ranges = extract_date_ranges(text)
    
    periods = []
    
    current_date = datetime.now()
    
    for start_month, start_year, end_value in date_ranges:
        
        start_date = datetime(
            int(start_year),
            int(start_month),
            1
        )
        
        if end_value == "current":
            end_date = current_date
        else:
            end_month, end_year = end_value.split()
            
            end_date = datetime(
                int(end_year),
                int(end_month),
                1
            )
        
        if end_date >= start_date:
            periods.append(
                (start_date, end_date)
            )
    
    if not periods:
        return 0.0
    
    # Sort by starting date
    periods.sort(key=lambda x: x[0])
    
    # Merge overlapping periods
    merged_periods = [periods[0]]
    
    for start, end in periods[1:]:
        
        last_start, last_end = merged_periods[-1]
        
        if start <= last_end:
            
            if end > last_end:
                merged_periods[-1] = (
                    last_start,
                    end
                )
        
        else:
            merged_periods.append(
                (start, end)
            )
    
    total_months = 0
    
    for start, end in merged_periods:
        
        months = (
            (end.year - start.year) * 12
            + (end.month - start.month)
        )
        
        total_months += months
    
    return round(total_months / 12, 1)

In [23]:
text = df.loc[
    df["ID"] == 22323967,
    "clean_resume"
].iloc[0]

print(calculate_total_experience(text))

16.7


In [24]:
df["calculated_experience"] = df["clean_resume"].apply(
    calculate_total_experience
)

In [25]:
df[
    ["ID", "Category", "years_experience", "calculated_experience"]
].head(20)

,ID,Category,years_experience,calculated_experience
0,16852973,HR,15,0.0
1,22323967,HR,0,16.7
2,33176873,HR,20,24.0
3,27018550,HR,0,0.0
4,17812897,HR,0,0.0
5,11592605,HR,0,22.8
6,25824789,HR,0,33.7
7,15375009,HR,0,17.4
8,11847784,HR,15,19.2
9,32896934,HR,0,41.6


In [26]:
df["calculated_experience"].describe()

count    2481.000000
mean       11.101814
std        14.588477
min         0.000000
25%         0.000000
50%         1.200000
75%        20.700000
max       157.300000
Name: calculated_experience, dtype: float64

In [27]:
comparison = df[
    ["ID", "years_experience", "calculated_experience"]
].copy()

comparison.head(20)

,ID,years_experience,calculated_experience
0,16852973,15,0.0
1,22323967,0,16.7
2,33176873,20,24.0
3,27018550,0,0.0
4,17812897,0,0.0
5,11592605,0,22.8
6,25824789,0,33.7
7,15375009,0,17.4
8,11847784,15,19.2
9,32896934,0,41.6


In [28]:
comparison[
    comparison["years_experience"] != comparison["calculated_experience"]
].head(20)

,ID,years_experience,calculated_experience
0,16852973,15,0.0
1,22323967,0,16.7
2,33176873,20,24.0
5,11592605,0,22.8
6,25824789,0,33.7
7,15375009,0,17.4
8,11847784,15,19.2
9,32896934,0,41.6
10,29149998,3,20.2
11,11480899,0,22.8


In [30]:
df["calculated_experience"] = df["clean_resume"].apply(
    calculate_total_experience
)

In [31]:
df["calculated_experience"].describe()

count    2481.000000
mean        9.362152
std        11.431150
min         0.000000
25%         0.000000
50%         1.000000
75%        18.200000
max        51.100000
Name: calculated_experience, dtype: float64

In [32]:
df[
    ["ID", "Category", "calculated_experience"]
].sort_values(
    "calculated_experience",
    ascending=False
).head(20)

,ID,Category,calculated_experience
2014,39027764,CONSTRUCTION,51.1
1996,12654876,CONSTRUCTION,47.1
932,12341902,AGRICULTURE,46.3
1153,20176584,CONSULTANT,46.1
1327,15790602,AUTOMOBILE,45.3
1811,33527446,ACCOUNTANT,44.7
687,18365443,HEALTHCARE,43.7
1095,16280971,SALES,43.2
1568,39295103,FINANCE,43.0
293,18159866,INFORMATION-TECHNOLOGY,42.7


In [33]:
df[
    ["ID", "Category", "calculated_experience"]
].head(20)

,ID,Category,calculated_experience
0,16852973,HR,0.0
1,22323967,HR,16.7
2,33176873,HR,24.0
3,27018550,HR,0.0
4,17812897,HR,0.0
5,11592605,HR,14.2
6,25824789,HR,33.7
7,15375009,HR,17.3
8,11847784,HR,19.0
9,32896934,HR,41.6


In [34]:
top_experience = df[
    ["ID", "Category", "calculated_experience", "clean_resume"]
].sort_values(
    "calculated_experience",
    ascending=False
)

top_experience[
    ["ID", "Category", "calculated_experience"]
].head(20)

,ID,Category,calculated_experience
2014,39027764,CONSTRUCTION,51.1
1996,12654876,CONSTRUCTION,47.1
932,12341902,AGRICULTURE,46.3
1153,20176584,CONSULTANT,46.1
1327,15790602,AUTOMOBILE,45.3
1811,33527446,ACCOUNTANT,44.7
687,18365443,HEALTHCARE,43.7
1095,16280971,SALES,43.2
1568,39295103,FINANCE,43.0
293,18159866,INFORMATION-TECHNOLOGY,42.7


In [35]:
print(top_experience["clean_resume"].iloc[0])

director of development professional summary real estate   development   construction executive versatile executive with 38 years experience in the real estate and construction industries possessing a combination of hands on technical experience coupled with strong organizational and leadership skills to manage project teams from concept to completion  honest and thoughtful style results in loyal teams that work toward a common goal  strategic thinker and planner with analytical skills and proven aptitude to manage complex and numerous projects and coordinate the efforts of corporate departments  landlords  designers  consultants  and contractors to turn over successful projects  honest and pragmatic approach to conflict resolution and problem solving  immense pursuit and tireless effort to produce successful and timely results  strong real estate aptitude allows for thorough due diligence and market analysis  for contract  purchase and lease administration  commercial and residential 

In [36]:
print(
    "Above 30 years:",
    (df["calculated_experience"] > 30).sum()
)

print(
    "Above 40 years:",
    (df["calculated_experience"] > 40).sum()
)

print(
    "Above 50 years:",
    (df["calculated_experience"] > 50).sum()
)

Above 30 years: 134
Above 40 years: 25
Above 50 years: 1


In [37]:
median_experience = df[
    ["ID", "Category", "calculated_experience"]
].sort_values(
    "calculated_experience"
)

median_experience.iloc[
    len(median_experience) // 2
]

ID                       38860712
Category                 ADVOCATE
calculated_experience         1.0
Name: 457, dtype: object

In [38]:
candidate = df[
    df["ID"] == 39027764
]

print(candidate["clean_resume"].iloc[0])

director of development professional summary real estate   development   construction executive versatile executive with 38 years experience in the real estate and construction industries possessing a combination of hands on technical experience coupled with strong organizational and leadership skills to manage project teams from concept to completion  honest and thoughtful style results in loyal teams that work toward a common goal  strategic thinker and planner with analytical skills and proven aptitude to manage complex and numerous projects and coordinate the efforts of corporate departments  landlords  designers  consultants  and contractors to turn over successful projects  honest and pragmatic approach to conflict resolution and problem solving  immense pursuit and tireless effort to produce successful and timely results  strong real estate aptitude allows for thorough due diligence and market analysis  for contract  purchase and lease administration  commercial and residential 

In [39]:
high_experience = df[
    df["calculated_experience"] > 40
][
    ["ID", "Category", "calculated_experience"]
].sort_values(
    "calculated_experience",
    ascending=False
)

high_experience

,ID,Category,calculated_experience
2014,39027764,CONSTRUCTION,51.1
1996,12654876,CONSTRUCTION,47.1
932,12341902,AGRICULTURE,46.3
1153,20176584,CONSULTANT,46.1
1327,15790602,AUTOMOBILE,45.3
1811,33527446,ACCOUNTANT,44.7
687,18365443,HEALTHCARE,43.7
1095,16280971,SALES,43.2
1568,39295103,FINANCE,43.0
293,18159866,INFORMATION-TECHNOLOGY,42.7


In [40]:
for _, row in high_experience.head(10).iterrows():
    
    print("=" * 80)
    print("ID:", row["ID"])
    print("Category:", row["Category"])
    print("Experience:", row["calculated_experience"])
    
    ranges = extract_date_ranges(
        df.loc[
            df["ID"] == row["ID"],
            "clean_resume"
        ].iloc[0]
    )
    
    print("Date ranges:")
    for r in ranges:
        print(r)

ID: 39027764
Category: CONSTRUCTION
Experience: 51.1
Date ranges:
('05', '1999', 'current')
('06', '1996', '05 1999')
('12', '1992', '06 1996')
('02', '1989', '12 1992')
('02', '1986', '02 1989')
('06', '1983', '02 1986')
('10', '1980', '06 1983')
('08', '1975', '10 1980')
ID: 12654876
Category: CONSTRUCTION
Experience: 47.1
Date ranges:
('08', '2015', 'current')
('01', '2012', '01 2015')
('01', '2012', '01 2015')
('01', '2007', '01 2008')
('01', '1998', '01 1999')
('01', '1994', '01 2012')
('01', '1990', '01 1994')
('01', '1986', '01 1990')
('01', '1983', '01 1986')
('01', '1979', '01 1983')
ID: 12341902
Category: AGRICULTURE
Experience: 46.3
Date ranges:
('10', '2007', 'current')
('08', '1999', '03 2015')
('03', '1982', '06 1999')
('01', '1975', '01 1977')
ID: 20176584
Category: CONSULTANT
Experience: 46.1
Date ranges:
('02', '2014', 'current')
('08', '2013', '12 2013')
('01', '2012', '08 2013')
('11', '2010', '11 2011')
('01', '2010', '03 2010')
('01', '2004', '10 2009')
('07', '200

In [41]:
required_experience = 3

def calculate_experience_score(experience, required):
    
    if required <= 0:
        return 0
    
    score = (experience / required) * 100
    
    return min(score, 100)

In [42]:
df["experience_score"] = df[
    "calculated_experience"
].apply(
    lambda x: calculate_experience_score(
        x,
        required_experience
    )
)

In [43]:
df[
    [
        "ID",
        "Category",
        "calculated_experience",
        "experience_score"
    ]
].sort_values(
    "experience_score",
    ascending=False
).head(20)

,ID,Category,calculated_experience,experience_score
1,22323967,HR,16.7,100.0
2466,49646155,AVIATION,23.6,100.0
2467,33627938,AVIATION,17.2,100.0
2471,17008356,AVIATION,9.2,100.0
2475,17655479,AVIATION,26.2,100.0
2476,12043694,AVIATION,11.7,100.0
2478,24589765,AVIATION,28.1,100.0
19,13879043,HR,6.4,100.0
22,25676643,HR,14.9,100.0
24,87968870,HR,21.8,100.0


In [44]:
df[
    [
        "ID",
        "Category",
        "calculated_experience",
        "experience_score"
    ]
].to_csv(
    "../data/processed/experience_results.csv",
    index=False
)